In [1]:
import os, glob
names = "main noattn fullattn temporal spatial nocls nounc reconly r4 r64 nodwt".split()
for n in names:
    for f in glob.glob(f"results/{n}_*.json"): os.remove(f)
if os.path.exists("results/cnnae_0.json"): os.remove("results/cnnae_0.json")
print(sorted(os.listdir("results")))

['.ipynb_checkpoints', 'cnnae_1.json', 'cnnae_2.json', 'lstmae_0.json', 'lstmae_1.json', 'lstmae_2.json', 'pca_0.json', 'supcnn_0.json', 'supcnn_1.json', 'supcnn_2.json']


# Cell 2: Check if it is in correct folder or not

In [2]:
import os
print(os.getcwd())
print(sorted(os.listdir()))      # gen.py, models.py, run.py, build_data.py, results ... dikhne chahiye

C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code
['.ipynb_checkpoints', '__pycache__', 'aggregate.py', 'build_data.py', 'data_cache.pkl', 'dwt_sweep.csv', 'dwt_sweep.py', 'gen.py', 'latency.json', 'latency.py', 'models.py', 'results', 'results.md', 'run.py', 'running_all_file.ipynb', 'scaling.json', 'scaling.py', 'sup_long.py']


# Cell 3: Make data (of ~15 sec, 800 MB data_cache.pkl is made)

In [3]:
%run build_data.py

raw generated 6.342568635940552
prep dwt True 11.945850849151611
prep dwt False 13.595311403274536
{'train': (500, 512, 64), 'val': (150, 512, 64), 'sup': (600, 512, 64), 'test_A': (500, 512, 64), 'calib_A': (100, 512, 64), 'test_B': (500, 512, 64), 'calib_B': (100, 512, 64), 'test_C': (500, 512, 64), 'calib_C': (100, 512, 64)}
       count      mean       std       min       25%      50%      75%  \
label                                                                    
0        0.0       NaN       NaN       NaN       NaN      NaN      NaN   
1      250.0  4.587481  5.574917 -4.937272 -0.242967  4.12462  9.53522   

            max  
label            
0           NaN  
1      14.77632  


# Cell 5: Small test (some minute)

In [4]:
%env EPOCHS=3
%run run.py --jobs main:0 --threads 4

env: EPOCHS=3
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
main 0 done in 12s A auroc 0.878


# Cell 6: poora experiment (mere 2 cores par ~50-60 min tha, aapke machine par kam lag sakta hai)

In [5]:
%env EPOCHS=12
variants = "noattn fullattn temporal spatial nocls nounc reconly r4 r64 nodwt".split()
jobs  = [f"{b}:{s}" for b in ("cnnae", "lstmae", "supcnn") for s in range(3)] + ["pca:0"]
jobs += [f"main:{s}" for s in range(3)]
jobs += [f"{v}:{s}" for v in variants for s in range(2)]
%run run.py --jobs {",".join(jobs)} --threads 4

env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
cnnae 0 done in 2s A auroc 0.865
main 1 done in 13s A auroc 0.899
main 2 done in 13s A auroc 0.881
noattn 0 done in 9s A auroc 0.892
noattn 1 done in 9s A auroc 0.883
fullattn 0 done in 11s A auroc 0.892
fullattn 1 done in 11s A auroc 0.881
temporal 0 done in 12s A auroc 0.890
temporal 1 done in 12s A auroc 0.889
spatial 0 done in 11s A auroc 0.897
spatial 1 done in 12s A auroc 0.889
nocls 0 done in 12s A auroc 0.877
nocls 1 done in 13s A auroc 0.888
nounc 0 done in 12s A auroc 0.870
nounc 1 done in 12s A auroc 0.869
reconly 0 done in 9s A auroc 0.869
reconly 1 done in 9s A auroc 0.869
r4 0 done in 14s A auroc 0.891
r4 1 done in 13s A auroc 0.894
r64 0 done in 14s A auroc 0.888
r64 1 done in 13s A auroc 0.883
nodwt 0 done in 13s A auroc 0.928
nodwt 1 done in 13s A auroc 0.939


# Cell 7: latency aur attention scaling

In [6]:
%run latency.py
%run scaling.py

DAS-ULRANet (MDLRA r=16) {'params': 45364, 'ms': 16.103750007459894}
DAS-ULRANet (MDLRA r=4) {'params': 45364, 'ms': 16.510650020791218}
DAS-ULRANet (MDLRA r=64) {'params': 45364, 'ms': 17.761449969839305}
Dual-axis full softmax attention {'params': 45364, 'ms': 15.10640000924468}
No attention {'params': 30516, 'ms': 10.555249988101423}
CNN-AE {'params': 230336, 'ms': 1.872749999165535}
BiLSTM-AE {'params': 205408, 'ms': 1.8343999981880188}
prep ms 1.269450003746897
{'S': 64, 'lowrank_r16': 5.1, 'full_softmax': 4.3}
{'S': 256, 'lowrank_r16': 17.5, 'full_softmax': 24.7}
{'S': 1024, 'lowrank_r16': 73.0, 'full_softmax': 288.5}
{'S': 2048, 'lowrank_r16': 164.2, 'full_softmax': 1251.2}
{'S': 4096, 'lowrank_r16': 538.2, 'full_softmax': 5517.7}


In [7]:
import os
os.remove("results/main_0.json")
%env EPOCHS=12
%run run.py --jobs main:0 --threads 4
%run aggregate.py

env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
main 0 done in 14s A auroc 0.886
# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.869 | 0.893 | 0.907 | 0.584 | 0.710 | 0.060 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.865 ± 0.000 | 0.887 ± 0.000 | 0.899 ± 0.000 | 0.571 ± 0.002 | 0.698 ± 0.002 | 0.064 ± 0.000 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.869 ± 0.000 | 0.895 ± 0.000 | 0.938 ± 0.003 | 0.588 ± 0.000 | 0.723 ± 0.001 | 0.039 ± 0.002 |
| 1-D CNN, supervised (label reference) | 3 | 0.825 ± 0.011 | 0.878 ± 0.008 | 0.960 ± 0.017 | 0.636 ± 0.014 | 0.765 ± 0.014 | 0.

In [8]:
variants = "noattn fullattn temporal spatial nocls nounc reconly r4 r64 nodwt".split()
jobs = ",".join(f"{v}:2" for v in variants)
print(jobs)
%env EPOCHS=12
%run run.py --jobs {jobs} --threads 4

noattn:2,fullattn:2,temporal:2,spatial:2,nocls:2,nounc:2,reconly:2,r4:2,r64:2,nodwt:2
env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
noattn 2 done in 9s A auroc 0.888
fullattn 2 done in 11s A auroc 0.883
temporal 2 done in 12s A auroc 0.888
spatial 2 done in 12s A auroc 0.882
nocls 2 done in 13s A auroc 0.893
nounc 2 done in 12s A auroc 0.871
reconly 2 done in 9s A auroc 0.868
r4 2 done in 13s A auroc 0.883
r64 2 done in 13s A auroc 0.879
nodwt 2 done in 13s A auroc 0.939


In [9]:
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.869 | 0.893 | 0.907 | 0.584 | 0.710 | 0.060 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.865 ± 0.000 | 0.887 ± 0.000 | 0.899 ± 0.000 | 0.571 ± 0.002 | 0.698 ± 0.002 | 0.064 ± 0.000 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.869 ± 0.000 | 0.895 ± 0.000 | 0.938 ± 0.003 | 0.588 ± 0.000 | 0.723 ± 0.001 | 0.039 ± 0.002 |
| 1-D CNN, supervised (label reference) | 3 | 0.825 ± 0.011 | 0.878 ± 0.008 | 0.960 ± 0.017 | 0.636 ± 0.014 | 0.765 ± 0.014 | 0.027 ± 0.012 |
| DAS-ULRANet (ours, full) | 3 | 0.889 ± 0.009 | 0.914 ± 0.009 | 0.935 ± 0.007 | 0

In [10]:
jobs = ",".join(f"{v}:{s}" for v in variants + ["main"] for s in (3, 4))
%run run.py --jobs {jobs} --threads 4

device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
noattn 3 done in 9s A auroc 0.898
noattn 4 done in 9s A auroc 0.887
fullattn 3 done in 12s A auroc 0.897
fullattn 4 done in 12s A auroc 0.883
temporal 3 done in 12s A auroc 0.896
temporal 4 done in 12s A auroc 0.891
spatial 3 done in 12s A auroc 0.901
spatial 4 done in 12s A auroc 0.893
nocls 3 done in 13s A auroc 0.896
nocls 4 done in 13s A auroc 0.894
nounc 3 done in 12s A auroc 0.875
nounc 4 done in 12s A auroc 0.869
reconly 3 done in 9s A auroc 0.869
reconly 4 done in 9s A auroc 0.869
r4 3 done in 13s A auroc 0.894
r4 4 done in 14s A auroc 0.896
r64 3 done in 14s A auroc 0.893
r64 4 done in 14s A auroc 0.891
nodwt 3 done in 13s A auroc 0.951
nodwt 4 done in 13s A auroc 0.928
main 3 done in 13s A auroc 0.894
main 4 done in 13s A auroc 0.900


In [11]:
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.869 | 0.893 | 0.907 | 0.584 | 0.710 | 0.060 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.865 ± 0.000 | 0.887 ± 0.000 | 0.899 ± 0.000 | 0.571 ± 0.002 | 0.698 ± 0.002 | 0.064 ± 0.000 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.869 ± 0.000 | 0.895 ± 0.000 | 0.938 ± 0.003 | 0.588 ± 0.000 | 0.723 ± 0.001 | 0.039 ± 0.002 |
| 1-D CNN, supervised (label reference) | 3 | 0.825 ± 0.011 | 0.878 ± 0.008 | 0.960 ± 0.017 | 0.636 ± 0.014 | 0.765 ± 0.014 | 0.027 ± 0.012 |
| DAS-ULRANet (ours, full) | 5 | 0.892 ± 0.008 | 0.917 ± 0.008 | 0.937 ± 0.008 | 0

# Cell 8: saari tables

In [12]:
s = open("aggregate.py", encoding="utf-8").read()
s = s.replace('open("results.md", "w")', 'open("results.md", "w", encoding="utf-8")')
open("aggregate.py", "w", encoding="utf-8").write(s)
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.869 | 0.893 | 0.907 | 0.584 | 0.710 | 0.060 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.865 ± 0.000 | 0.887 ± 0.000 | 0.899 ± 0.000 | 0.571 ± 0.002 | 0.698 ± 0.002 | 0.064 ± 0.000 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.869 ± 0.000 | 0.895 ± 0.000 | 0.938 ± 0.003 | 0.588 ± 0.000 | 0.723 ± 0.001 | 0.039 ± 0.002 |
| 1-D CNN, supervised (label reference) | 3 | 0.825 ± 0.011 | 0.878 ± 0.008 | 0.960 ± 0.017 | 0.636 ± 0.014 | 0.765 ± 0.014 | 0.027 ± 0.012 |
| DAS-ULRANet (ours, full) | 5 | 0.892 ± 0.008 | 0.917 ± 0.008 | 0.937 ± 0.008 | 0

In [13]:
import pickle, torch, numpy as np, gen, models
print("torch", torch.__version__)
D = pickle.load(open("data_cache.pkl", "rb"))
X = D[True]["train"][0]
print("NaN in data:", np.isnan(X).any(), X.dtype, X.shape)
torch.manual_seed(0)
m = models.ULRANet(512, 64, r=16, attn="none")
opt = torch.optim.Adam(m.parameters(), lr=2e-3)
Xt = torch.from_numpy(X)
for step in range(60):
    j = (step * 16) % 480
    x = Xt[j:j + 16]
    rec, z, ev = m(x)
    loss = ((rec - x) ** 2).mean()
    opt.zero_grad(); loss.backward()
    gn = torch.nn.utils.clip_grad_norm_(m.parameters(), 5.)
    opt.step()
    if step % 5 == 0 or not torch.isfinite(loss):
        print(step, float(loss), float(gn), bool(torch.isfinite(z).all()))
    if not torch.isfinite(loss): break

torch 2.11.0+cu128
NaN in data: False float32 (500, 512, 64)


C:\Users\fiza\AppData\Local\Temp\ipykernel_36588\311472891.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print(step, float(loss), float(gn), bool(torch.isfinite(z).all()))


0 0.5700029730796814 2.9774105548858643 True
5 0.19334755837917328 0.5961546301841736 True
10 0.1479269564151764 0.3305823504924774 True
15 0.13996142148971558 0.16512684524059296 True
20 0.127043679356575 0.08871611207723618 True
25 0.11954542249441147 0.08272568881511688 True
30 0.11749126762151718 0.06436377763748169 True
35 0.11401839554309845 0.03732422739267349 True
40 0.11733189225196838 0.03388448432087898 True
45 0.12908867001533508 0.02514939196407795 True
50 0.12255317717790604 0.018574604764580727 True
55 0.11650921404361725 0.01557738333940506 True


# run small test on dwt_sweep

In [14]:
%env EPOCHS=3
%run dwt_sweep.py --seeds 0

env: EPOCHS=3
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
raw data ready in 7 s
none               seed 0 | dev_A 0.876 dev_B 0.919 | test_A 0.864 (4 s)
db4_L4_soft_k1.0   seed 0 | dev_A 0.868 dev_B 0.848 | test_A 0.879 (4 s)
db4_L4_soft_k0.5   seed 0 | dev_A 0.893 dev_B 0.896 | test_A 0.894 (4 s)
db4_L4_soft_k0.25  seed 0 | dev_A 0.893 dev_B 0.902 | test_A 0.881 (4 s)
db4_L4_hard_k1.0   seed 0 | dev_A 0.894 dev_B 0.897 | test_A 0.892 (4 s)
db4_L4_hard_k0.5   seed 0 | dev_A 0.900 dev_B 0.930 | test_A 0.888 (4 s)
db4_L2_soft_k1.0   seed 0 | dev_A 0.866 dev_B 0.904 | test_A 0.862 (5 s)
db4_L3_soft_k1.0   seed 0 | dev_A 0.869 dev_B 0.906 | test_A 0.874 (4 s)
sym4_L4_soft_k0.5  seed 0 | dev_A 0.901 dev_B 0.900 | test_A 0.898 (4 s)
haar_L4_soft_k0.5  seed 0 | dev_A 0.898 dev_B 0.851 | test_A 0.895 (4 s)

Sorted by DEV (mean of dev_A and dev_B). Choose the config from the dev_mean column, then read the TEST columns.

                   dev_A  dev_B  test_A   F1_A  FAR_A  test_B  test_C  

In [19]:
%env EPOCHS=12
%run dwt_sweep.py --seeds 0,1,2

env: EPOCHS=12
epochs: 12
raw data ready in 9 s
none               seed 0 | dev_A 0.947 dev_B 0.940 | test_A 0.925 (14 s)
none               seed 1 | dev_A 0.942 dev_B 0.928 | test_A 0.939 (13 s)
none               seed 2 | dev_A 0.941 dev_B 0.904 | test_A 0.937 (14 s)
db4_L4_soft_k1.0   seed 0 | dev_A 0.875 dev_B 0.860 | test_A 0.891 (14 s)
db4_L4_soft_k1.0   seed 1 | dev_A 0.874 dev_B 0.843 | test_A 0.888 (14 s)
db4_L4_soft_k1.0   seed 2 | dev_A 0.870 dev_B 0.877 | test_A 0.882 (14 s)
db4_L4_soft_k0.5   seed 0 | dev_A 0.908 dev_B 0.942 | test_A 0.909 (14 s)
db4_L4_soft_k0.5   seed 1 | dev_A 0.912 dev_B 0.958 | test_A 0.915 (14 s)
db4_L4_soft_k0.5   seed 2 | dev_A 0.910 dev_B 0.947 | test_A 0.914 (14 s)
db4_L4_soft_k0.25  seed 0 | dev_A 0.921 dev_B 0.968 | test_A 0.915 (14 s)
db4_L4_soft_k0.25  seed 1 | dev_A 0.922 dev_B 0.964 | test_A 0.915 (14 s)
db4_L4_soft_k0.25  seed 2 | dev_A 0.921 dev_B 0.968 | test_A 0.910 (14 s)
db4_L4_hard_k1.0   seed 0 | dev_A 0.904 dev_B 0.898 | test_A 0.9

In [20]:
import run
print(run.E)

12


### Ek patch cell chalayein. Ye gen.py, build_data.py, run.py aur aggregate.py badalta hai. Ye Prep(dwt=True) ko k=0.15 par set karta hai, cache mein ek teesri entry ("univ", purana k=1.0) jodta hai, aur ablation mein dwtuniv variant jodta hai. Teeno baselines bhi D[True] use karte hain, isliye unhe bhi naya preprocessing mil jayega.

In [23]:
def patch(fn, pairs):
    s = open(fn, encoding="utf-8").read()
    for old, new in pairs:
        assert old in s, (fn, old[:60])
        s = s.replace(old, new, 1)
    open(fn, "w", encoding="utf-8").write(s)

patch("gen.py", [
 ('def dwt_denoise(X, wavelet="db4", level=4):',
  'DWT_K = 0.15   # chosen on DEV sets (dwt_sweep.py): thr = k * sigma * sqrt(2 ln W)\n\ndef dwt_denoise(X, wavelet="db4", level=4, mode="soft", k=DWT_K):'),
 ('thr = sigma * np.sqrt(2 * np.log(Xt.shape[-1]))',
  'thr = k * sigma * np.sqrt(2 * np.log(Xt.shape[-1]))'),
 ('pywt.threshold(c, thr, mode="soft")', 'pywt.threshold(c, thr, mode=mode)'),
 ('def __init__(self, dwt=True): self.dwt = dwt',
  'def __init__(self, dwt=True, k=DWT_K): self.dwt, self.k = dwt, k'),
 ('return dwt_denoise(Z) if self.dwt else', 'return dwt_denoise(Z, k=self.k) if self.dwt else'),
])

patch("build_data.py", [
 ('''for dwt in (True,False):
    P=gen.Prep(dwt=dwt).fit(Xtr)
    out[dwt]={k:(P(v[0]),v[1]) for k,v in D.items()}
    print("prep dwt",dwt,time.time()-t)''',
  '''for key, kw in ((True, dict(dwt=True)), (False, dict(dwt=False)), ("univ", dict(dwt=True, k=1.0))):
    P=gen.Prep(**kw).fit(Xtr)
    out[key]={k:(P(v[0]),v[1]) for k,v in D.items()}
    print("prep", key, time.time()-t)'''),
])

patch("run.py", [
 ('"nodwt":    dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=False),',
  '"nodwt":    dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=False),\n "dwtuniv":  dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt="univ"),'),
])

patch("aggregate.py", [
 ('("nodwt", "No DWT denoising"),',
  '("nodwt", "No DWT denoising"), ("dwtuniv", "DWT, universal threshold (k=1.0)"),'),
])
print("patched")

AssertionError: ('gen.py', 'def dwt_denoise(X, wavelet="db4", level=4):')

In [1]:
# 1) purane results hatayein (naye preprocessing par ye sab invalid hain)
import glob, os
for f in glob.glob("results/*.json"): os.remove(f)

In [2]:
# 2) data dobara banayein (teen entries: True, False, "univ")
%run build_data.py

raw generated 8.215741157531738
prep True 13.338913202285767
prep False 15.241195678710938
prep univ 20.86276602745056
{'train': (500, 512, 64), 'val': (150, 512, 64), 'sup': (600, 512, 64), 'test_A': (500, 512, 64), 'calib_A': (100, 512, 64), 'test_B': (500, 512, 64), 'calib_B': (100, 512, 64), 'test_C': (500, 512, 64), 'calib_C': (100, 512, 64)}
       count      mean       std       min       25%      50%      75%  \
label                                                                    
0        0.0       NaN       NaN       NaN       NaN      NaN      NaN   
1      250.0  4.587481  5.574917 -4.937272 -0.242967  4.12462  9.53522   

            max  
label            
0           NaN  
1      14.77632  


In [3]:
# 3) poora experiment, 5 seeds
%env EPOCHS=12
variants = "noattn fullattn temporal spatial nocls nounc reconly r4 r64 nodwt dwtuniv".split()
jobs = [f"{b}:{s}" for b in ("cnnae","lstmae","supcnn") for s in range(3)] + ["pca:0"]
jobs += [f"{v}:{s}" for v in variants + ["main"] for s in range(5)]
jobs = ",".join(jobs)
%run run.py --jobs {jobs} --threads 4

env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
cnnae 0 done in 12s A auroc 0.825
cnnae 1 done in 2s A auroc 0.828
cnnae 2 done in 2s A auroc 0.828
lstmae 0 done in 2s A auroc 0.843
lstmae 1 done in 2s A auroc 0.843
lstmae 2 done in 2s A auroc 0.843
supcnn 0 done in 2s A auroc 0.884
supcnn 1 done in 2s A auroc 0.910
supcnn 2 done in 2s A auroc 0.893
pca 0 done in 1s A auroc 0.836
noattn 0 done in 14s A auroc 0.901
noattn 1 done in 9s A auroc 0.905
noattn 2 done in 9s A auroc 0.909
noattn 3 done in 9s A auroc 0.923
noattn 4 done in 9s A auroc 0.918
fullattn 0 done in 12s A auroc 0.917
fullattn 1 done in 12s A auroc 0.927
fullattn 2 done in 12s A auroc 0.919
fullattn 3 done in 12s A auroc 0.916
fullattn 4 done in 12s A auroc 0.923
temporal 0 done in 12s A auroc 0.923
temporal 1 done in 12s A auroc 0.902
temporal 2 done in 12s A auroc 0.908
temporal 3 done in 12s A auroc 0.922
temporal 4 done in 12s A auroc 0.918
spatial 0 done in 12s A auroc 0.913
spatial 1 done in 11s A a

In [4]:
# 4) tables
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.836 | 0.866 | 0.966 | 0.448 | 0.612 | 0.016 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.827 ± 0.002 | 0.857 ± 0.001 | 0.961 ± 0.009 | 0.428 ± 0.011 | 0.592 ± 0.009 | 0.017 ± 0.005 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.843 ± 0.000 | 0.872 ± 0.000 | 0.967 ± 0.000 | 0.472 ± 0.000 | 0.634 ± 0.000 | 0.016 ± 0.000 |
| 1-D CNN, supervised (label reference) | 3 | 0.895 ± 0.013 | 0.923 ± 0.010 | 0.956 ± 0.027 | 0.684 ± 0.034 | 0.797 ± 0.018 | 0.032 ± 0.022 |
| DAS-ULRANet (ours, full) | 5 | 0.924 ± 0.009 | 0.946 ± 0.007 | 0.984 ± 0.007 | 0

# patch (isko chalane ke baad Kernel → Restart)

In [7]:
def patch(fn, pairs):
    s = open(fn, encoding="utf-8").read(); changed = False
    for old, new in pairs:
        if new in s:     print(f"{fn}: already patched -> {old[:50]!r}"); continue
        if old not in s: print(f"{fn}: NOT FOUND       -> {old[:60]!r}"); continue
        s = s.replace(old, new, 1); changed = True; print(f"{fn}: patched         -> {old[:50]!r}")
    if changed: open(fn, "w", encoding="utf-8").write(s)

patch("build_data.py", [
 ('D["train"]=(Xtr,mtr); D["val"]=(Xval,mval); D["sup"]=(Xsup,msup)',
  'D["train"]=(Xtr,mtr); D["val"]=(Xval,mval); D["sup"]=(Xsup,msup)\nD["dev_A"]=gen.make_dataset("A",250,250,seed=40); D["dev_B"]=gen.make_dataset("B",250,250,seed=41)'),
])

patch("run.py", [
 ('def job(name, seed, D):\n    t0 = time.time()',
  'def job(name, seed, D):\n    t0 = time.time()\n    base = name[:-3] if name.endswith("_nd") else name      # "_nd" = baseline on data WITHOUT DWT\n    key = False if name.endswith("_nd") else True'),
 ('elif name in ("cnnae", "lstmae"):\n        data = D[True]; m, hist = train_ae(name, data["train"][0], seed)',
  'elif base in ("cnnae", "lstmae"):\n        data = D[key]; m, hist = train_ae(base, data["train"][0], seed)'),
 ('elif name == "supcnn":\n        data = D[True]; Xs, ms = data["sup"]',
  'elif base == "supcnn":\n        data = D[key]; Xs, ms = data["sup"]'),
 ('elif name == "pca":\n        data = D[True]; Z =',
  'elif base == "pca":\n        data = D[key]; Z ='),
 ('        cache = {}\n        def parts(X):',
  '        cache = {}\n        fw = {"w": 0.5}                                         # weight of the reconstruction score\n        def parts(X):'),
 ('return 0.5 * s_rec + 0.5 * p', 'return fw["w"] * s_rec + (1 - fw["w"]) * p'),
 ('        res = eval_all(score_fn, data)\n        extra = {}\n',
  '        if combined and "dev_A" in data:\n'
  '            best = None\n'
  '            for w in (0.0, 0.25, 0.5, 0.75, 1.0):\n'
  '                fw["w"] = w\n'
  '                a_dev = np.mean([auroc(data[k][1].label.values, score_fn(data[k][0])) for k in ("dev_A", "dev_B")])\n'
  '                if best is None or a_dev > best[0]: best = (a_dev, w)\n'
  '            fw["w"] = best[1]\n'
  '        res = eval_all(score_fn, data)\n        extra = {"fusion_w": fw["w"]}\n'),
 ('extra["components_auroc_A"] = comp', 'extra["components_auroc_A"] = comp; extra["fusion_w"] = fw["w"]'),
])

patch("aggregate.py", [
 ('("supcnn", "1-D CNN, supervised (label reference)"), ("main", "DAS-ULRANet (ours, full)")]',
  '("supcnn", "1-D CNN, supervised (label reference)"),\n         ("pca_nd", "PCA, no DWT"), ("cnnae_nd", "CNN-AE, no DWT"), ("lstmae_nd", "BiLSTM-AE, no DWT"), ("supcnn_nd", "1-D CNN supervised, no DWT"),\n         ("main", "DAS-ULRANet (ours, full)"), ("nodwt", "DAS-ULRANet, no DWT")]'),
])

build_data.py: already patched -> 'D["train"]=(Xtr,mtr); D["val"]=(Xval,mval); D["sup'
run.py: already patched -> 'def job(name, seed, D):\n    t0 = time.time()'
run.py: already patched -> 'elif name in ("cnnae", "lstmae"):\n        data = D'
run.py: already patched -> 'elif name == "supcnn":\n        data = D[True]; Xs,'
run.py: already patched -> 'elif name == "pca":\n        data = D[True]; Z ='
run.py: already patched -> '        cache = {}\n        def parts(X):'
run.py: already patched -> 'return 0.5 * s_rec + 0.5 * p'
run.py: already patched -> '        res = eval_all(score_fn, data)\n        ext'
run.py: already patched -> 'extra["components_auroc_A"] = comp'
aggregate.py: already patched -> '("supcnn", "1-D CNN, supervised (label reference)"'


# Restart ke baad ye cells (patch cell dobara nahi)

In [1]:
# Cell 2: purane ULRANet results hatayein (baselines purane rehne dein), purani tables ka backup
import glob, os, shutil
shutil.copy("results.md", "results_round1.md")
redo = "noattn fullattn temporal spatial nocls r4 r64 nodwt dwtuniv main".split()
for n in redo:
    for f in glob.glob(f"results/{n}_*.json"): os.remove(f)

In [2]:
# Cell 3: cache dobara banayein (ab dev_A, dev_B ke saath)
%run build_data.py

raw generated 12.28640103340149
prep True 20.66493797302246
prep False 22.95550513267517
prep univ 29.933369636535645
{'train': (500, 512, 64), 'val': (150, 512, 64), 'sup': (600, 512, 64), 'dev_A': (500, 512, 64), 'dev_B': (500, 512, 64), 'test_A': (500, 512, 64), 'calib_A': (100, 512, 64), 'test_B': (500, 512, 64), 'calib_B': (100, 512, 64), 'test_C': (500, 512, 64), 'calib_C': (100, 512, 64)}
       count      mean       std       min       25%      50%      75%  \
label                                                                    
0        0.0       NaN       NaN       NaN       NaN      NaN      NaN   
1      250.0  4.587481  5.574917 -4.937272 -0.242967  4.12462  9.53522   

            max  
label            
0           NaN  
1      14.77632  


In [3]:
# Cell 4: naye baselines + ULRANet variants dobara (~50 jobs, ~12 min)
%env EPOCHS=12
redo = "noattn fullattn temporal spatial nocls r4 r64 nodwt dwtuniv main".split()
jobs = ["pca_nd:0"] + [f"{b}_nd:{s}" for b in ("cnnae", "lstmae", "supcnn") for s in range(3)]
jobs += [f"{v}:{s}" for v in redo for s in range(5)]
jobs = ",".join(jobs)
%run run.py --jobs {jobs} --threads 4

env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
pca_nd 0 done in 1s A auroc 0.799
cnnae_nd 0 done in 7s A auroc 0.794
cnnae_nd 1 done in 2s A auroc 0.794
cnnae_nd 2 done in 2s A auroc 0.797
lstmae_nd 0 done in 2s A auroc 0.801
lstmae_nd 1 done in 2s A auroc 0.801
lstmae_nd 2 done in 2s A auroc 0.801
supcnn_nd 0 done in 2s A auroc 0.869
supcnn_nd 1 done in 2s A auroc 0.875
supcnn_nd 2 done in 2s A auroc 0.845
noattn 0 done in 15s A auroc 0.948
noattn 1 done in 10s A auroc 0.948
noattn 2 done in 10s A auroc 0.932
noattn 3 done in 9s A auroc 0.957
noattn 4 done in 9s A auroc 0.958
fullattn 0 done in 12s A auroc 0.965
fullattn 1 done in 12s A auroc 0.977
fullattn 2 done in 12s A auroc 0.958
fullattn 3 done in 12s A auroc 0.852
fullattn 4 done in 12s A auroc 0.969
temporal 0 done in 12s A auroc 0.947
temporal 1 done in 12s A auroc 0.957
temporal 2 done in 12s A auroc 0.955
temporal 3 done in 12s A auroc 0.964
temporal 4 done in 12s A auroc 0.937
spatial 0 done in 12s A auroc 

In [4]:
# Cell 5: tables + dev par chune gaye fusion weights
%run aggregate.py
import json, glob, collections
W = collections.defaultdict(list)
for f in sorted(glob.glob("results/*.json")):
    r = json.load(open(f)); w = r["extra"].get("fusion_w")
    if w is not None: W[r["name"]].append(w)
for k, v in W.items(): print(k, v)

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.836 | 0.866 | 0.966 | 0.448 | 0.612 | 0.016 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.827 ± 0.002 | 0.857 ± 0.001 | 0.961 ± 0.009 | 0.428 ± 0.011 | 0.592 ± 0.009 | 0.017 ± 0.005 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.843 ± 0.000 | 0.872 ± 0.000 | 0.967 ± 0.000 | 0.472 ± 0.000 | 0.634 ± 0.000 | 0.016 ± 0.000 |
| 1-D CNN, supervised (label reference) | 3 | 0.895 ± 0.013 | 0.923 ± 0.010 | 0.956 ± 0.027 | 0.684 ± 0.034 | 0.797 ± 0.018 | 0.032 ± 0.022 |
| PCA, no DWT | 1 | 0.799 | 0.829 | 0.955 | 0.340 | 0.501 | 0.016 |
| CNN-AE, no D

# Cell 1: patch (chalane ke baad Kernel → Restart, kyunki gen badla hai)

In [5]:
def patch(fn, pairs):
    s = open(fn, encoding="utf-8").read(); changed = False
    for old, new in pairs:
        if new in s:     print(f"{fn}: already patched -> {old[:50]!r}"); continue
        if old not in s: print(f"{fn}: NOT FOUND       -> {old[:60]!r}"); continue
        s = s.replace(old, new, 1); changed = True; print(f"{fn}: patched         -> {old[:50]!r}")
    if changed: open(fn, "w", encoding="utf-8").write(s)

patch("gen.py", [
 ('def inject_pseudo(Z, rng, fs=200):', 'def inject_pseudo(Z, rng, fs=200, kinds=("bump", "step", "noise")):'),
 ('kind = rng.choice(["bump", "step", "noise"])', 'kind = rng.choice(list(kinds))'),
])
patch("run.py", [
 ('gen.inject_pseudo(xn[:BS // 2].numpy(), rng)', 'gen.inject_pseudo(xn[:BS // 2].numpy(), rng, kinds=vcfg.get("kinds", ("bump", "step", "noise")))'),
 ('"nodwt":    dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=False),',
  '"nodwt":    dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=False),\n'
  ' "pa_step":  dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=True, kinds=("step",)),\n'
  ' "pa_bump":  dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=True, kinds=("bump",)),\n'
  ' "pa_noise": dict(attn="mdlra", r=16, lam=(1, .2, .2), dwt=True, kinds=("noise",)),'),
 ('    elif base == "supcnn":\n        data = D[key]; Xs, ms = data["sup"]\n        m, hist = train_ae("supcnn", None, seed, sup_data=(Xs, ms.label.values.astype(np.float32)))',
  '    elif base in ("supcnn", "supcnn_pa"):\n        data = D[key]\n'
  '        if base == "supcnn_pa":      # same supervision as ULRANet: normal + injected pseudo-anomalies, no real leak labels\n'
  '            Ztr = data["train"][0]; Xs = np.concatenate([Ztr, gen.inject_pseudo(Ztr, np.random.default_rng(seed + 100))])\n'
  '            ys = np.concatenate([np.zeros(len(Ztr)), np.ones(len(Ztr))]).astype(np.float32)\n'
  '        else:\n'
  '            Xs, ms = data["sup"]; ys = ms.label.values.astype(np.float32)\n'
  '        m, hist = train_ae("supcnn", None, seed, sup_data=(Xs, ys))'),
])
patch("aggregate.py", [
 ('("supcnn_nd", "1-D CNN supervised, no DWT"),',
  '("supcnn_nd", "1-D CNN supervised, no DWT"), ("supcnn_pa", "1-D CNN trained on pseudo-anomalies (no real labels)"),'),
 ('("r4", "Rank r=4"),',
  '("pa_step", "Pseudo-anomalies: step family only"), ("pa_bump", "Pseudo-anomalies: bump family only"), ("pa_noise", "Pseudo-anomalies: noise-burst family only"),\n      ("r4", "Rank r=4"),'),
])

gen.py: patched         -> 'def inject_pseudo(Z, rng, fs=200):'
gen.py: patched         -> 'kind = rng.choice(["bump", "step", "noise"])'
run.py: patched         -> 'gen.inject_pseudo(xn[:BS // 2].numpy(), rng)'
run.py: patched         -> '"nodwt":    dict(attn="mdlra", r=16, lam=(1, .2, .'
run.py: patched         -> '    elif base == "supcnn":\n        data = D[key]; '
aggregate.py: patched         -> '("supcnn_nd", "1-D CNN supervised, no DWT"),'
aggregate.py: patched         -> '("r4", "Rank r=4"),'


# Restart ke baad (~4 minute)

In [1]:
%env EPOCHS=12
jobs = [f"{v}:{s}" for v in ("pa_step", "pa_bump", "pa_noise") for s in range(3)] + [f"supcnn_pa:{s}" for s in range(3)]
jobs = ",".join(jobs)
%run run.py --jobs {jobs} --threads 4

env: EPOCHS=12
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
pa_step 0 done in 17s A auroc 0.843
pa_step 1 done in 12s A auroc 0.843
pa_step 2 done in 13s A auroc 0.843
pa_bump 0 done in 12s A auroc 0.893
pa_bump 1 done in 13s A auroc 0.902
pa_bump 2 done in 12s A auroc 0.917
pa_noise 0 done in 16s A auroc 0.845
pa_noise 1 done in 16s A auroc 0.844
pa_noise 2 done in 17s A auroc 0.856
supcnn_pa 0 done in 4s A auroc 0.928
supcnn_pa 1 done in 4s A auroc 0.906
supcnn_pa 2 done in 4s A auroc 0.904


In [2]:
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.836 | 0.866 | 0.966 | 0.448 | 0.612 | 0.016 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.827 ± 0.002 | 0.857 ± 0.001 | 0.961 ± 0.009 | 0.428 ± 0.011 | 0.592 ± 0.009 | 0.017 ± 0.005 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.843 ± 0.000 | 0.872 ± 0.000 | 0.967 ± 0.000 | 0.472 ± 0.000 | 0.634 ± 0.000 | 0.016 ± 0.000 |
| 1-D CNN, supervised (label reference) | 3 | 0.895 ± 0.013 | 0.923 ± 0.010 | 0.956 ± 0.027 | 0.684 ± 0.034 | 0.797 ± 0.018 | 0.032 ± 0.022 |
| PCA, no DWT | 1 | 0.799 | 0.829 | 0.955 | 0.340 | 0.501 | 0.016 |
| CNN-AE, no D

In [3]:
import json, glob, collections
W = collections.defaultdict(list)
for f in sorted(glob.glob("results/pa_*.json")):
    r = json.load(open(f)); W[r["name"]].append(r["extra"].get("fusion_w"))
print(dict(W))

{'pa_bump': [0.5, 0.25, 0.5], 'pa_noise': [1.0, 1.0, 1.0], 'pa_step': [1.0, 1.0, 1.0]}


# otway_hybrid.py made

In [7]:
import os
root = r"C:/Users/fiza/DAS_Datasets/data"
for d in sorted(os.listdir(root)):
    p = os.path.join(root, d)
    if os.path.isdir(p):
        n = sum(1 for dp, _, fs in os.walk(p) for f in fs
                if f.lower().endswith((".sgy", ".segy")) and "__MACOSX" not in dp and not f.startswith("._"))
        print(f"{d:30s} SEG-Y files: {n}")

SIMFIP dataset                 SEG-Y files: 0
otway_metadata                 SEG-Y files: 0
otway_microseismic             SEG-Y files: 80
synthetic_das                  SEG-Y files: 0


# Step 1 (1 minute): files aur wells check

In [10]:
%run otway_hybrid.py --data "C:/Users/fiza/DAS_Datasets/data/otway_microseismic" --inspect

skip (header only / empty): event_2_5_strain_geom.sgy
80 files, wells [3, 4, 5, 6, 7], windows per well: {3: 336, 4: 416, 5: 405, 6: 384, 7: 400}, raw window array (1941, 512, 64), 21s
first file: C:/Users/fiza/DAS_Datasets/data/otway_microseismic\RAW_seismograms\RAW_seismograms\event_10_3_strain_geom.sgy -> (channels, time) = (1346, 5000) fs = 1000.0


# Step 2 (~2 minute): smoke test

In [11]:
%env EPOCHS=12
%run otway_hybrid.py --data "C:/Users/fiza/DAS_Datasets/data/otway_microseismic" --folds 0 --seeds 0

env: EPOCHS=12
skip (header only / empty): event_2_5_strain_geom.sgy
80 files, wells [3, 4, 5, 6, 7], windows per well: {3: 336, 4: 416, 5: 405, 6: 384, 7: 400}, raw window array (1941, 512, 64), 21s
normalised: bg std 1.15, ev std 1.37 (should be ~1)
device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
epochs: 12
fold 0: test wells A/B=3, C=4; train 892, val 297, test_A 252, test_B 252, test_C 316
  fold 0 main      seed 0  AUROC  A(inj) 0.913  B(real events) 0.549  C(inj, 2nd well) 0.922  (24s)


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 0 nodwt     seed 0  AUROC  A(inj) 0.920  B(real events) 0.569  C(inj, 2nd well) 0.907  (24s)


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 0 cnnae     seed 0  AUROC  A(inj) 0.774  B(real events) 0.580  C(inj, 2nd well) 0.753  (4s)
  fold 0 lstmae    seed 0  AUROC  A(inj) 0.750  B(real events) 0.525  C(inj, 2nd well) 0.774  (7s)
  fold 0 pca       seed 0  AUROC  A(inj) 0.785  B(real events) 0.589  C(inj, 2nd well) 0.770  (4s)
  fold 0 supcnn_pa seed 0  AUROC  A(inj) 0.899  B(real events) 0.382  C(inj, 2nd well) 0.880  (19s)
| Method | n runs | AUROC A (injected leak) | AUROC C (2nd held-out well) | AUROC B (real microseismic events) | F1 A (recal. thr) | FAR A (recal. thr) |
|---|---|---|---|---|---|---|
| pca | 1 | 0.785 ± 0.000 | 0.770 ± 0.000 | 0.589 ± 0.000 | 0.187 ± 0.000 | 0.000 ± 0.000 |
| cnnae | 1 | 0.774 ± 0.000 | 0.753 ± 0.000 | 0.580 ± 0.000 | 0.133 ± 0.000 | 0.000 ± 0.000 |
| lstmae | 1 | 0.750 ± 0.000 | 0.774 ± 0.000 | 0.525 ± 0.000 | 0.200 ± 0.000 | 0.000 ± 0.000 |
| supcnn_pa | 1 | 0.899 ± 0.000 | 0.880 ± 0.000 | 0.382 ± 0.000 | 0.781 ± 0.000 | 0.016 ± 0.000 |
| nodwt | 1 | 0.920 ± 0.000 | 0.907 ± 0.

# Step 3 (~10 minute): poora run

In [12]:
%run otway_hybrid.py --data "C:/Users/fiza/DAS_Datasets/data/otway_microseismic" --folds 0,1,2,3,4 --seeds 0,1

loaded cache (1941, 512, 64)
normalised: bg std 1.15, ev std 1.37 (should be ~1)
epochs: 12
fold 0: test wells A/B=3, C=4; train 892, val 297, test_A 252, test_B 252, test_C 316


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 0 main      seed 1  AUROC  A(inj) 0.947  B(real events) 0.552  C(inj, 2nd well) 0.937  (66s)


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 0 nodwt     seed 1  AUROC  A(inj) 0.947  B(real events) 0.543  C(inj, 2nd well) 0.936  (64s)
  fold 0 cnnae     seed 1  AUROC  A(inj) 0.772  B(real events) 0.581  C(inj, 2nd well) 0.752  (10s)
  fold 0 lstmae    seed 1  AUROC  A(inj) 0.738  B(real events) 0.532  C(inj, 2nd well) 0.757  (10s)
  fold 0 pca       seed 1  AUROC  A(inj) 0.785  B(real events) 0.589  C(inj, 2nd well) 0.770  (4s)
  fold 0 supcnn_pa seed 1  AUROC  A(inj) 0.879  B(real events) 0.436  C(inj, 2nd well) 0.859  (21s)
fold 1: test wells A/B=4, C=5; train 840, val 280, test_A 316, test_B 316, test_C 305


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


... (11 outputs left out)

C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 4 main      seed 0  AUROC  A(inj) 0.962  B(real events) 0.648  C(inj, 2nd well) 0.958  (66s)


C:\Users\fiza\Documents\datavis\Paper_code_biLSTM\das_ulranet_experiment_code\run.py:31: RuntimeWarning: invalid value encountered in scalar divide
  return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


  fold 4 nodwt     seed 0  AUROC  A(inj) 0.951  B(real events) 0.615  C(inj, 2nd well) 0.949  (66s)
  fold 4 cnnae     seed 0  AUROC  A(inj) 0.840  B(real events) 0.676  C(inj, 2nd well) 0.801  (10s)
  fold 4 lstmae    seed 0  AUROC  A(inj) 0.815  B(real events) 0.665  C(inj, 2nd well) 0.837  (11s)
  fold 4 pca       seed 0  AUROC  A(inj) 0.851  B(real events) 0.673  C(inj, 2nd well) 0.831  (4s)
  fold 4 supcnn_pa seed 0  AUROC  A(inj) 0.957  B(real events) 0.657  C(inj, 2nd well) 0.929  (19s)
  fold 4 main      seed 1  AUROC  A(inj) 0.965  B(real events) 0.573  C(inj, 2nd well) 0.939  (65s)
  fold 4 nodwt     seed 1  AUROC  A(inj) 0.954  B(real events) 0.616  C(inj, 2nd well) 0.945  (65s)
  fold 4 cnnae     seed 1  AUROC  A(inj) 0.840  B(real events) 0.678  C(inj, 2nd well) 0.801  (10s)
  fold 4 lstmae    seed 1  AUROC  A(inj) 0.811  B(real events) 0.688  C(inj, 2nd well) 0.827  (10s)
  fold 4 pca       seed 1  AUROC  A(inj) 0.851  B(real events) 0.673  C(inj, 2nd well) 0.831  (4s)
  

In [13]:
%run otway_hybrid.py --data "C:/Users/fiza/DAS_Datasets/data/otway_microseismic" --folds 0,1,2,3,4 --seeds 0,1

loaded cache (1941, 512, 64)
normalised: bg std 1.15, ev std 1.37 (should be ~1)
epochs: 12
fold 0: test wells A/B=3, C=4; train 892, val 297, test_A 252, test_B 252, test_C 316
fold 1: test wells A/B=4, C=5; train 840, val 280, test_A 316, test_B 316, test_C 305
fold 2: test wells A/B=5, C=6; train 864, val 288, test_A 305, test_B 304, test_C 288
fold 3: test wells A/B=6, C=7; train 868, val 289, test_A 288, test_B 288, test_C 300
fold 4: test wells A/B=7, C=3; train 904, val 301, test_A 300, test_B 300, test_C 252
| Method | n runs | AUROC A (injected leak) | AUROC C (2nd held-out well) | AUROC B (real microseismic events) | F1 A (recal. thr) | FAR A (recal. thr) |
|---|---|---|---|---|---|---|
| pca | 10 | 0.790 ± 0.046 | 0.808 ± 0.059 | 0.636 ± 0.053 | 0.282 ± 0.189 | 0.017 ± 0.016 |
| cnnae | 10 | 0.772 ± 0.050 | 0.782 ± 0.054 | 0.624 ± 0.049 | 0.226 ± 0.178 | 0.015 ± 0.014 |
| lstmae | 10 | 0.781 ± 0.045 | 0.803 ± 0.044 | 0.608 ± 0.101 | 0.302 ± 0.197 | 0.020 ± 0.018 |
| supcnn_p

In [14]:
%run stats_report.py

# Significance report (AUROC differences, reference = DAS-ULRANet 'main')

### Synthetic benchmark: A = in-domain, B/C = zero-shot sites (Welch, over seeds)

| vs | site | mean diff (ref - other) | 95% bootstrap CI | p | n |
|---|---|---|---|---|---|
| cnnae | A | +0.138 | [+0.129, +0.149] | 9.84e-06 | 5/3 |
| cnnae | C | +0.097 | [+0.081, +0.117] | 0.000851 | 5/3 |
| cnnae_nd | A | +0.170 | [+0.160, +0.180] | 4.36e-06 | 5/3 |
| cnnae_nd | C | +0.138 | [+0.122, +0.158] | 0.000224 | 5/3 |
| dwtuniv | A | +0.076 | [+0.062, +0.091] | 2.13e-05 | 5/5 |
| dwtuniv | C | +0.009 | [-0.010, +0.032] | 0.481 | 5/5 |
| fullattn | A | +0.021 | [-0.011, +0.069] | 0.431 | 5/5 |
| fullattn | C | +0.013 | [-0.020, +0.050] | 0.551 | 5/5 |
| lstmae | A | +0.122 | [+0.112, +0.132] | 2.57e-05 | 5/3 |
| lstmae | C | +0.082 | [+0.066, +0.102] | 0.00165 | 5/3 |
| lstmae_nd | A | +0.164 | [+0.154, +0.174] | 7.86e-06 | 5/3 |
| lstmae_nd | C | +0.135 | [+0.119, +0.155] | 0.000246 | 5/3 |
| noattn | A | +0.016 | [

In [16]:
%run gpu_cost.py

device: cuda NVIDIA GeForce RTX 5070 Laptop GPU
DAS-ULRANet (MDLRA r=16) {'params': 45364, 'ms_batch1': 15.665450016967952, 'ms_per_window_batch64': 1.005460937449243}
DAS-ULRANet (MDLRA r=4) {'params': 45364, 'ms_batch1': 34.51065003173426, 'ms_per_window_batch64': 0.4110953132112627}
DAS-ULRANet (MDLRA r=64) {'params': 45364, 'ms_batch1': 37.75879996828735, 'ms_per_window_batch64': 0.8012132811927586}
Dual-axis full softmax attention {'params': 45364, 'ms_batch1': 25.13820002786815, 'ms_per_window_batch64': 0.7632453125552274}
No attention {'params': 30516, 'ms_batch1': 11.87124999705702, 'ms_per_window_batch64': 0.47137578076217324}
CNN-AE {'params': 230336, 'ms_batch1': 6.434999988414347, 'ms_per_window_batch64': 0.07476562495867256}
BiLSTM-AE {'params': 205408, 'ms_batch1': 10.083500004839152, 'ms_per_window_batch64': 0.168600000506558}
{'S': 64, 'lowrank_r16': 5.19, 'lowrank_r16_peakMB': 28, 'full_softmax': 7.53, 'full_softmax_peakMB': 28}
{'S': 256, 'lowrank_r16': 31.09, 'lowran

In [17]:
print(open("stats_report.md", encoding="utf-8").read())
import json
print(json.dumps(json.load(open("gpu_cost.json")), indent=1))

# Significance report (AUROC differences, reference = DAS-ULRANet 'main')

### Synthetic benchmark: A = in-domain, B/C = zero-shot sites (Welch, over seeds)

| vs | site | mean diff (ref - other) | 95% bootstrap CI | p | n |
|---|---|---|---|---|---|
| cnnae | A | +0.138 | [+0.129, +0.149] | 9.84e-06 | 5/3 |
| cnnae | C | +0.097 | [+0.081, +0.117] | 0.000851 | 5/3 |
| cnnae_nd | A | +0.170 | [+0.160, +0.180] | 4.36e-06 | 5/3 |
| cnnae_nd | C | +0.138 | [+0.122, +0.158] | 0.000224 | 5/3 |
| dwtuniv | A | +0.076 | [+0.062, +0.091] | 2.13e-05 | 5/5 |
| dwtuniv | C | +0.009 | [-0.010, +0.032] | 0.481 | 5/5 |
| fullattn | A | +0.021 | [-0.011, +0.069] | 0.431 | 5/5 |
| fullattn | C | +0.013 | [-0.020, +0.050] | 0.551 | 5/5 |
| lstmae | A | +0.122 | [+0.112, +0.132] | 2.57e-05 | 5/3 |
| lstmae | C | +0.082 | [+0.066, +0.102] | 0.00165 | 5/3 |
| lstmae_nd | A | +0.164 | [+0.154, +0.174] | 7.86e-06 | 5/3 |
| lstmae_nd | C | +0.135 | [+0.119, +0.155] | 0.000246 | 5/3 |
| noattn | A | +0.016 | [

In [19]:
%run aggregate.py

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.836 | 0.866 | 0.966 | 0.448 | 0.612 | 0.016 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.827 ± 0.002 | 0.857 ± 0.001 | 0.961 ± 0.009 | 0.428 ± 0.011 | 0.592 ± 0.009 | 0.017 ± 0.005 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.843 ± 0.000 | 0.872 ± 0.000 | 0.967 ± 0.000 | 0.472 ± 0.000 | 0.634 ± 0.000 | 0.016 ± 0.000 |
| 1-D CNN, supervised (label reference) | 3 | 0.895 ± 0.013 | 0.923 ± 0.010 | 0.956 ± 0.027 | 0.684 ± 0.034 | 0.797 ± 0.018 | 0.032 ± 0.022 |
| PCA, no DWT | 1 | 0.799 | 0.829 | 0.955 | 0.340 | 0.501 | 0.016 |
| CNN-AE, no D

In [20]:
print(open("results.md", encoding="utf-8").read())

# DAS-ULRANet: experiment results (synthetic benchmark)

All numbers are produced by `run.py` in this folder. Values: mean ± std over seeds (n shown). Threshold θ = 99th percentile of validation-normal scores (label-free). Test sets: 250 normal + 250 leak windows per site.

## Table 1. In-domain performance (Site A)

| Method | n | AUROC | AUPRC | Precision | Recall | F1 | FAR |
|---|---|---|---|---|---|---|---|
| PCA (linear AE, k=20) | 1 | 0.836 | 0.866 | 0.966 | 0.448 | 0.612 | 0.016 |
| CNN-AE (unsup. adaptation of CNN) | 3 | 0.827 ± 0.002 | 0.857 ± 0.001 | 0.961 ± 0.009 | 0.428 ± 0.011 | 0.592 ± 0.009 | 0.017 ± 0.005 |
| BiLSTM-AE (unsup. adaptation of BiLSTM) | 3 | 0.843 ± 0.000 | 0.872 ± 0.000 | 0.967 ± 0.000 | 0.472 ± 0.000 | 0.634 ± 0.000 | 0.016 ± 0.000 |
| 1-D CNN, supervised (label reference) | 3 | 0.895 ± 0.013 | 0.923 ± 0.010 | 0.956 ± 0.027 | 0.684 ± 0.034 | 0.797 ± 0.018 | 0.032 ± 0.022 |
| PCA, no DWT | 1 | 0.799 | 0.829 | 0.955 | 0.340 | 0.501 | 0.016 |
| CNN-AE, no D